# Standardize Orig Transaction Reports to Buttecounty Schema

This notebook loads the two Orig TRANSACTION_DETAIL_REPORT files, standardizes their columns to match **buttecounty_07_2022_cleaned.csv** (reference), so they can be merged with the buttecounty files.

**Reference (target columns):** `timestamp`, `numeric_user_id`, `numeric_pass_id`, `fare_id`, `fare_type`, `pass_name`, `rider_type`, `vehicle_number`, `stop_code`, `stop_id`, `stop_name`, `stop_lat`, `stop_lon`, `trip_id`, `gtfs_route_id`, `route_id`, `route_long_name`

In [ ]:
import pandas as pd
import numpy as np
import os

# Paths: set PROJECT_ROOT to your project folder (where 'data' lives)
PROJECT_ROOT = r"C:\Users\zelaskar\Box\2024 Zakir Elaskar"
BASE = os.path.join(PROJECT_ROOT, "data", "Transaction data")
OUT_DIR = BASE
REFERENCE_FILE = os.path.join(BASE, "buttecounty_07_2022_cleaned.csv")
ORIG_OCT_DEC_2022 = os.path.join(BASE, "Orig TRANSACTION_DETAIL_REPORT_(OCTOBER_1_2022-DECEMBER_31_2022)_cleaned.csv")
ORIG_Q3_2023 = os.path.join(BASE, "Orig TRANSACTION_DETAIL_REPORT_Q3 2023_cleaned.csv")

# Reference schema (buttecounty columns)
REF_COLUMNS = [
    "timestamp", "numeric_user_id", "numeric_pass_id", "fare_id", "fare_type", "pass_name",
    "rider_type", "vehicle_number", "stop_code", "stop_id", "stop_name", "stop_lat", "stop_lon",
    "trip_id", "gtfs_route_id", "route_id", "route_long_name"
]

In [ ]:
def load_orig_report(path: str, source_name: str) -> pd.DataFrame:
    """Load an Orig TRANSACTION_DETAIL_REPORT CSV. Skip metadata rows; use row 4 as header. Drop footer rows."""
    df = pd.read_csv(path, header=4, low_memory=False)
    # Drop rows where Date Time is missing or not parseable (removes footer/metadata)
    df["Date Time"] = pd.to_datetime(df["Date Time"], errors="coerce")
    df = df.dropna(subset=["Date Time"]).copy()
    df["_source_file"] = source_name
    return df


def standardize_orig_to_buttecounty(df: pd.DataFrame) -> pd.DataFrame:
    """Map Orig report columns to buttecounty schema. Returns DataFrame with REF_COLUMNS + source_file."""
    out = pd.DataFrame()
    dt = pd.to_datetime(df["Date Time"])
    out["timestamp"] = dt.apply(lambda t: f"{t.month}/{t.day}/{t.year} {t.hour}:{t.minute}")
    card = df.get("Card ID / Sequence #", pd.Series(dtype=object)).fillna("").astype(str)
    out["numeric_user_id"] = card.str.extract(r"(\\d+)", expand=False)
    out["numeric_pass_id"] = ""
    prod = df.get("Product", pd.Series(dtype=object)).fillna("").astype(str)
    out["fare_id"] = prod.str.lower().str.replace(r"\\s+", "_", regex=True)
    out["fare_type"] = df.get("Transaction Type", pd.Series(dtype=object)).fillna("").astype(str)
    out["pass_name"] = prod
    out["rider_type"] = "Regular"
    bus = df.get("Bus", pd.Series(dtype=object)).astype(str)
    out["vehicle_number"] = bus.str.replace(r"\.0$", "", regex=True)
    out["stop_code"] = df.get("Stop Abbr", pd.Series(dtype=object)).fillna("").astype(str).str.replace(r"\.0$", "", regex=True)
    out["stop_id"] = df.get("Stop ID", pd.Series(dtype=object)).fillna("").astype(str).str.replace(r"\.0$", "", regex=True)
    out["stop_name"] = df.get("Stop Name", pd.Series(dtype=object)).fillna("").astype(str)
    out["stop_lat"] = pd.to_numeric(df.get("Latitude", pd.Series(dtype=object)), errors="coerce")
    out["stop_lon"] = pd.to_numeric(df.get("Longitude", pd.Series(dtype=object)), errors="coerce")
    out["trip_id"] = ""
    out["gtfs_route_id"] = ""
    route = df.get("Route", pd.Series(dtype=object)).astype(str).str.replace(r"\.0$", "", regex=True)
    out["route_id"] = route
    out["route_long_name"] = "Route " + route.replace("nan", "Unknown")
    out["source_file"] = df["_source_file"]
    return out[[c for c in REF_COLUMNS + ["source_file"] if c in out.columns]]

In [ ]:
# Load both Orig files, standardize to buttecounty schema, and save (optional: merge)
orig_oct_dec = load_orig_report(ORIG_OCT_DEC_2022, "Orig TRANSACTION_DETAIL_REPORT_(OCTOBER_1_2022-DECEMBER_31_2022)_cleaned.csv")
orig_q3_2023 = load_orig_report(ORIG_Q3_2023, "Orig TRANSACTION_DETAIL_REPORT_Q3 2023_cleaned.csv")

std_oct_dec = standardize_orig_to_buttecounty(orig_oct_dec)
std_q3_2023 = standardize_orig_to_buttecounty(orig_q3_2023)

print("Orig Oct-Dec 2022: rows loaded", len(orig_oct_dec), "-> standardized", len(std_oct_dec))
print("Orig Q3 2023 (Jan-Mar): rows loaded", len(orig_q3_2023), "-> standardized", len(std_q3_2023))
print("Standardized columns:", std_oct_dec.columns.tolist())
std_oct_dec.head(3)

In [ ]:
# Save standardized files (same schema as buttecounty; ready to merge)
std_oct_dec.to_csv(os.path.join(OUT_DIR, "Orig_OCT_DEC_2022_standardized.csv"), index=False)
std_q3_2023.to_csv(os.path.join(OUT_DIR, "Orig_Q3_2023_standardized.csv"), index=False)
print("Saved: Orig_OCT_DEC_2022_standardized.csv, Orig_Q3_2023_standardized.csv")

### Merge all transaction files into one

List the exact file names to merge; only these files are loaded (no pattern matching).

In [ ]:
# Transaction files to merge (exact names; only these are loaded)
TRANSACTION_FILES = [
    "buttecounty_07_2022_cleaned.csv",
    "buttecounty_08_2022_cleaned.csv",
    "buttecounty_09_2022_cleaned.csv",
    "buttecounty_10_2022_cleaned.csv",
    "buttecounty_11_2022_cleaned.csv",
    "buttecounty_12_2022_cleaned.csv",
    "Orig_OCT_DEC_2022_standardized.csv",
    "Orig_Q3_2023_standardized.csv",
]

# Load each file by name from BASE
cols = REF_COLUMNS + ["source_file"]
dfs = []
for f in TRANSACTION_FILES:
    path = os.path.join(BASE, f)
    df = pd.read_csv(path, low_memory=False)
    if "source_file" not in df.columns:
        df["source_file"] = f
    for c in cols:
        if c not in df.columns:
            df[c] = ""
    dfs.append(df[cols])

# Concatenate and save
combined = pd.concat(dfs, ignore_index=True)
out_path = os.path.join(PROJECT_ROOT, "data", "Combined datasets", "Transactions_combined.csv")
os.makedirs(os.path.dirname(out_path), exist_ok=True)
combined.to_csv(out_path, index=False)
print("Loaded", len(TRANSACTION_FILES), "files ->", len(combined), "rows")
print("Saved:", out_path)
combined.head()